# RECAST Constraint Evaluation Framework

This notebook implements a standardized, plug-and-play evaluation pipeline for measuring
zero-shot constraint-following performance of LLMs on the RECAST-30K dataset.

**How to use:**
1. Upload `constraint_checker.py`, `viz_utils.py`, and `evaluator.py` to your Google Drive folder alongside this notebook
2. Fill in the configuration widgets in Section 0 (model name, HF token, Drive folder)
3. Run All Cells

## Expected file structure
```
MyDrive/<project_dir>/
├── data/                  # Put recast_30k.json here
│   └── recast_30k.json
├── output/                # Per-model results written here
│   └── <model_name>/
│       ├── <model>_responses.jsonl
│       └── plots/
├── scores/                # Cross-model comparison CSVs
│   ├── <model>_<timestamp>_metrics.json
│   ├── <model>_<timestamp>_per_type.csv
│   └── team_comparison.csv
├── constraint_checker.py  # Auto-copied from this repo
├── evaluator.py           # Auto-copied from this repo
├── viz_utils.py           # Auto-copied from this repo
└── judge.py               # Optional LLM judge module
```

Results are automatically named after the model via `MODEL_NAME_SAFE`, so multiple
models can be evaluated into the same Drive folder without conflicts. Section 7 merges
all available results into a single comparison table.

## Section 0: Setup & Config

Install dependencies and configure evaluation parameters via interactive widgets.
Fill in your model name, HuggingFace token, and dataset path, then run the next cell to lock in values.

In [ ]:
!pip install -q transformers datasets accelerate bitsandbytes torch tqdm pandas matplotlib seaborn huggingface_hub ipywidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display

MODEL_NAME_W    = widgets.Text(value="Qwen/Qwen2.5-1.5B-Instruct", description="HF Model ID:", style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
HF_TOKEN_W      = widgets.Password(value="", description="HF Token:", style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
GDRIVE_PROJECT_DIR_W = widgets.Text(value="MyDrive/recast_eval", description="Drive folder:", style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
USE_4BIT_W      = widgets.Checkbox(value=True, description="4-bit quant")
MAX_NEW_TOKENS_W = widgets.IntSlider(value=512, min=128, max=1024, description="Max tokens:", style={'description_width': 'initial'})
BATCH_SIZE_W    = widgets.IntSlider(value=4, min=1, max=16, description="Batch size:", style={'description_width': 'initial'})
NUM_SAMPLES_W   = widgets.IntText(value=0, description="Num samples (0=all):", style={'description_width': 'initial'})

display(widgets.VBox([
    MODEL_NAME_W, HF_TOKEN_W, GDRIVE_PROJECT_DIR_W,
    USE_4BIT_W, MAX_NEW_TOKENS_W, BATCH_SIZE_W, NUM_SAMPLES_W
]))

In [ ]:
import os
from datetime import datetime

# Read widget values into plain variables
MODEL_NAME     = MODEL_NAME_W.value
HF_TOKEN       = HF_TOKEN_W.value
GDRIVE_PROJECT_DIR = GDRIVE_PROJECT_DIR_W.value
USE_4BIT       = USE_4BIT_W.value
MAX_NEW_TOKENS = MAX_NEW_TOKENS_W.value
BATCH_SIZE     = BATCH_SIZE_W.value
NUM_SAMPLES    = NUM_SAMPLES_W.value

MODEL_NAME_SAFE = MODEL_NAME.replace("/", "_").replace("-", "_")
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Derived paths — consistent with driver.ipynb
DRIVE_BASE = f"/content/drive/{GDRIVE_PROJECT_DIR}"
DATA_DIR = f"{DRIVE_BASE}/data"
OUTPUT_DIR = f"{DRIVE_BASE}/output"
DATASET_PATH = f"{DATA_DIR}/recast_30k.json"
RESULTS_DIR = f"{OUTPUT_DIR}/{MODEL_NAME_SAFE}"
SCORES_DIR = f"{DRIVE_BASE}/scores"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SCORES_DIR, exist_ok=True)

SCORE_PREFIX = f"{MODEL_NAME_SAFE}_{RUN_TIMESTAMP}"

print(f"Model:        {MODEL_NAME}")
print(f"Model safe:   {MODEL_NAME_SAFE}")
print(f"Run timestamp:{RUN_TIMESTAMP}")
print(f"Project dir:  {DRIVE_BASE}")
print(f"Data dir:     {DATA_DIR}")
print(f"Dataset:      {DATASET_PATH}")
print(f"Output dir:   {OUTPUT_DIR}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Scores dir:   {SCORES_DIR}")
print(f"4-bit quant:  {USE_4BIT}")
print(f"Max tokens:   {MAX_NEW_TOKENS}")
print(f"Batch size:   {BATCH_SIZE}")
print(f"Num samples:  {NUM_SAMPLES if NUM_SAMPLES > 0 else 'all'}")

In [ ]:
# Copy helper modules to Drive so they persist across sessions
import shutil

for fname in ["constraint_checker.py", "viz_utils.py", "evaluator.py", "judge.py"]:
    src = f"/content/{fname}"
    dst = f"{DRIVE_BASE}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {fname} -> {dst}")
    elif os.path.exists(dst):
        shutil.copy2(dst, src)
        print(f"Restored {fname} from Drive")
    else:
        print(f"WARNING: {fname} not found in /content/ or Drive. Upload it to your Drive folder.")

# Ensure /content is on the path for imports
import sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

In [ ]:
# Environment check
import torch
import platform
import transformers

print(f"Python version:       {platform.python_version()}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name:             {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:           {torch.cuda.get_device_properties(0).total_mem / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Inference will be very slow.")

## Section 1: Google Drive Mount & Model Loading

Mounts Google Drive for persistent storage, then loads the model and tokenizer.
If you selected 4-bit quantization, the model is loaded with BitsAndBytes for T4 compatibility.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset exists
assert os.path.exists(DATASET_PATH), (
    f"Dataset not found at {DATASET_PATH}. "
    f"Upload recast_30k.json to {DATA_DIR}/"
)
print(f"Dataset found: {DATASET_PATH}")
print(f"Results directory ready: {RESULTS_DIR}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

# Login to HuggingFace if token provided
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to HuggingFace.")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

# Load model
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

model.eval()
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Section 2: Dataset Loading

Loads the RECAST-30K dataset from the local path specified in the config.
Parses each record to extract prompts, constraints, difficulty levels, and reference responses.
If `NUM_SAMPLES` is set, performs stratified sampling across difficulty levels.

In [ ]:
import json
import re
from collections import Counter, defaultdict
import random

# Load raw data
raw_data = []
if DATASET_PATH.endswith(".jsonl"):
    with open(DATASET_PATH, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                raw_data.append(json.loads(line))
else:
    with open(DATASET_PATH, "r") as f:
        loaded = json.load(f)
        if isinstance(loaded, list):
            raw_data = loaded
        elif isinstance(loaded, dict):
            # Try common wrapper keys
            for key in ["data", "instances", "examples", "records"]:
                if key in loaded and isinstance(loaded[key], list):
                    raw_data = loaded[key]
                    break
            if not raw_data:
                raw_data = [loaded]

print(f"Total raw records loaded: {len(raw_data)}")
print(f"Keys in first record: {list(raw_data[0].keys())}")
print()

# Constraint type patterns for parsing from prompt text
CONSTRAINT_PATTERNS = [
    r'(?i)(at least|at most|exactly|no more than|no fewer than)\s+(\d+)\s+(words?|sentences?|paragraphs?|bullet points?|sections?)',
    r'(?i)(must|should)\s+(start|begin)\s+with\s+["\']?(.+?)["\']?[\.,]',
    r'(?i)(must|should)\s+end\s+with\s+["\']?(.+?)["\']?[\.,]',
    r'(?i)(include|contain|use|mention)\s+(?:the\s+)?(?:words?|keywords?)\s*[:\-]?\s*(.+?)[\.;]',
    r'(?i)(?:do not|don\'t|avoid|never)\s+(?:use|include|mention)\s+(?:the\s+)?(?:words?|keywords?)\s*[:\-]?\s*(.+?)[\.;]',
    r'(?i)(?:write|respond|answer)\s+(?:entirely\s+)?in\s+(lowercase|uppercase|all caps)',
    r'(?i)(?:format|write)\s+(?:as|in|using)\s+(json|bullet points?|numbered list|markdown)',
    r'(?i)(?:do not|don\'t|avoid)\s+(?:use|include)\s+(?:any\s+)?(commas?|periods?)',
    r'(?i)(?:include|add|end with)\s+(?:a\s+)?(?:p\.?s\.?|postscript)',
]

def parse_constraints_from_prompt(prompt_text):
    """Attempt to extract constraint dicts from raw prompt text using regex."""
    constraints = []
    for pattern in CONSTRAINT_PATTERNS:
        matches = re.findall(pattern, prompt_text)
        for match in matches:
            constraints.append({"type": "parsed_from_prompt", "raw_match": str(match)})
    return constraints

def infer_difficulty(num_constraints):
    """Infer difficulty level from number of constraints."""
    if num_constraints <= 2:
        return "L1"
    elif num_constraints <= 4:
        return "L2"
    elif num_constraints <= 7:
        return "L3"
    else:
        return "L4"

def parse_record(record, idx):
    """Parse a raw record into standardized format."""
    # Extract prompt
    prompt = record.get("winner_prompt") or record.get("prompt") or record.get("instruction") or ""

    # Extract constraints
    constraints = record.get("constraints", None)
    if constraints is None:
        constraints = record.get("constraint_list", None)
    if constraints is None:
        # Try to parse from prompt text
        constraints = parse_constraints_from_prompt(prompt)
    if isinstance(constraints, str):
        try:
            constraints = json.loads(constraints)
        except json.JSONDecodeError:
            constraints = [{"type": "raw", "description": constraints}]
    if not isinstance(constraints, list):
        constraints = [constraints] if constraints else []

    # Extract difficulty level
    difficulty = record.get("difficulty_level") or record.get("difficulty") or record.get("level")
    if difficulty is None:
        difficulty = infer_difficulty(len(constraints))
    difficulty = str(difficulty)
    if not difficulty.startswith("L"):
        difficulty = f"L{difficulty}"

    # Extract reference response
    reference = record.get("winner_response") or record.get("response") or record.get("output") or ""

    return {
        "id": record.get("id", idx),
        "prompt": prompt,
        "constraints": constraints,
        "num_constraints": len(constraints),
        "difficulty_level": difficulty,
        "reference_response": reference
    }

# Parse all records
dataset = [parse_record(r, i) for i, r in enumerate(raw_data)]

# Stratified sampling if NUM_SAMPLES is set
if NUM_SAMPLES > 0:
    by_level = defaultdict(list)
    for item in dataset:
        by_level[item["difficulty_level"]].append(item)
    per_level = max(1, NUM_SAMPLES // len(by_level))
    sampled = []
    for level in sorted(by_level.keys()):
        items = by_level[level]
        random.seed(42)
        sampled.extend(random.sample(items, min(per_level, len(items))))
    dataset = sampled
    print(f"Stratified sampled down to {len(dataset)} instances")

# Print summary
level_counts = Counter(d["difficulty_level"] for d in dataset)
print(f"\nTotal parsed instances: {len(dataset)}")
print(f"Distribution by difficulty level:")
for level in sorted(level_counts.keys()):
    print(f"  {level}: {level_counts[level]}")

print(f"\n--- Sample 1 ---")
print(f"Prompt: {dataset[0]['prompt'][:200]}...")
print(f"Constraints ({dataset[0]['num_constraints']}): {dataset[0]['constraints'][:3]}")
print(f"Difficulty: {dataset[0]['difficulty_level']}")

print(f"\n--- Sample 2 ---")
print(f"Prompt: {dataset[1]['prompt'][:200]}...")
print(f"Constraints ({dataset[1]['num_constraints']}): {dataset[1]['constraints'][:3]}")
print(f"Difficulty: {dataset[1]['difficulty_level']}")

## Section 3: Inference

Runs the model on all dataset prompts using greedy decoding.
Checkpoints results to Drive every 50 batches and resumes from existing checkpoints automatically.

In [ ]:
from tqdm.auto import tqdm

checkpoint_path = f"{RESULTS_DIR}/{MODEL_NAME_SAFE}_responses.jsonl"

# Load existing checkpoint if present
completed_ids = set()
results = []
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                completed_ids.add(rec["id"])
                results.append(rec)
    print(f"Resumed from checkpoint: {len(completed_ids)} instances already completed.")

# Filter to remaining items
remaining = [d for d in dataset if d["id"] not in completed_ids]
print(f"Remaining instances to process: {len(remaining)}")

# Check if model supports chat template
has_chat_template = hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None

def format_prompt(prompt_text):
    """Format a prompt for the model."""
    if has_chat_template:
        messages = [{"role": "user", "content": prompt_text}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"User: {prompt_text}\nAssistant:"

# Process in batches
batch_count = 0
checkpoint_file = open(checkpoint_path, "a")

try:
    for i in tqdm(range(0, len(remaining), BATCH_SIZE), desc="Inference"):
        batch = remaining[i : i + BATCH_SIZE]
        prompts = [format_prompt(item["prompt"]) for item in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(model.device)

        input_len = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        # Decode only new tokens
        for j, item in enumerate(batch):
            generated_tokens = outputs[j][input_len:]
            response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

            result = {
                "id": item["id"],
                "prompt": item["prompt"],
                "response": response,
                "constraints": item["constraints"],
                "difficulty_level": item["difficulty_level"]
            }
            results.append(result)
            checkpoint_file.write(json.dumps(result) + "\n")

        batch_count += 1

        # Flush checkpoint every 50 batches
        if batch_count % 50 == 0:
            checkpoint_file.flush()
            print(f"  Checkpoint saved at batch {batch_count} ({len(results)} total)")

        # Clear cache every 100 batches
        if batch_count % 100 == 0:
            torch.cuda.empty_cache()

finally:
    checkpoint_file.close()

print(f"\nInference complete. Total results: {len(results)}")
torch.cuda.empty_cache()

## Section 4: Rule-Based Constraint Evaluation

Imports the `ConstraintChecker` class and runs constraint checking on all model responses.
Each constraint is evaluated independently and results are stored per-instance.

In [ ]:
from constraint_checker import ConstraintChecker

checker = ConstraintChecker()

for item in tqdm(results, desc="Evaluating constraints"):
    eval_result = checker.check_all(item["response"], item["constraints"])
    item.update(eval_result)

# Quick summary
checked = [r for r in results if r["num_checked"] > 0]
avg_csr = sum(r["per_constraint_csr"] for r in checked) / len(checked) if checked else 0
hard_rate = sum(1 for r in checked if r["hard_csr"]) / len(checked) if checked else 0
print(f"\nEvaluation complete.")
print(f"Instances with checkable constraints: {len(checked)}/{len(results)}")
print(f"Overall per-constraint CSR: {avg_csr:.4f}")
print(f"Overall hard CSR: {hard_rate:.4f}")

torch.cuda.empty_cache()

## Section 5: Compute Metrics

Computes per-constraint CSR and hard CSR by difficulty level and by constraint type.
Saves all score files to `scores/` with the model name and run timestamp in the filename.

In [ ]:
import pandas as pd

SCORE_PREFIX = f"{MODEL_NAME_SAFE}_{RUN_TIMESTAMP}"

# Group by difficulty level
by_level = defaultdict(list)
for r in results:
    if r["num_checked"] > 0:
        by_level[r["difficulty_level"]].append(r)

# Per-level metrics
metrics_by_level = {}
for level in ["L1", "L2", "L3", "L4"]:
    items = by_level.get(level, [])
    if items:
        csr = sum(r["per_constraint_csr"] for r in items) / len(items)
        hard = sum(1 for r in items if r["hard_csr"]) / len(items)
    else:
        csr, hard = 0.0, 0.0
    metrics_by_level[level] = {"csr": round(csr, 4), "hard_csr": round(hard, 4), "count": len(items)}

# Overall
all_checked = [r for r in results if r["num_checked"] > 0]
overall_csr = sum(r["per_constraint_csr"] for r in all_checked) / len(all_checked) if all_checked else 0
overall_hard = sum(1 for r in all_checked if r["hard_csr"]) / len(all_checked) if all_checked else 0
metrics_by_level["Overall"] = {"csr": round(overall_csr, 4), "hard_csr": round(overall_hard, 4), "count": len(all_checked)}

print("Per-constraint CSR and Hard CSR by difficulty level:")
for level in ["L1", "L2", "L3", "L4", "Overall"]:
    m = metrics_by_level[level]
    print(f"  {level}: CSR={m['csr']:.4f}, Hard CSR={m['hard_csr']:.4f} (n={m['count']})")

# Per-constraint-type pass rate
type_pass = defaultdict(lambda: {"passed": 0, "total": 0})
for r in results:
    for cr in r.get("results", []):
        if cr["passed"] is not None:
            type_pass[cr["type"]]["total"] += 1
            if cr["passed"]:
                type_pass[cr["type"]]["passed"] += 1

per_type_rates = {}
for ctype, counts in sorted(type_pass.items()):
    rate = counts["passed"] / counts["total"] if counts["total"] > 0 else 0
    per_type_rates[ctype] = round(rate, 4)
    print(f"  {ctype}: {rate:.4f} ({counts['passed']}/{counts['total']})")

# Save metrics JSON to scores/
metrics_output = {
    "model": MODEL_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "by_level": metrics_by_level,
    "per_type": per_type_rates
}
metrics_path = f"{SCORES_DIR}/{SCORE_PREFIX}_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics_output, f, indent=2)
print(f"\nSaved metrics to {metrics_path}")

# Save per-type CSV to scores/
per_type_df = pd.DataFrame([
    {"constraint_type": k, "pass_rate": v} for k, v in per_type_rates.items()
]).sort_values("pass_rate", ascending=False)
per_type_csv_path = f"{SCORES_DIR}/{SCORE_PREFIX}_per_type.csv"
per_type_df.to_csv(per_type_csv_path, index=False)
print(f"Saved per-type rates to {per_type_csv_path}")

# Save degradation row CSV to scores/
deg_row = {"Model": MODEL_NAME, "Run": RUN_TIMESTAMP}
for level in ["L1", "L2", "L3", "L4", "Overall"]:
    deg_row[f"{level}_CSR"] = metrics_by_level[level]["csr"]
    deg_row[f"{level}_Hard"] = metrics_by_level[level]["hard_csr"]
deg_df = pd.DataFrame([deg_row])
deg_path = f"{SCORES_DIR}/{SCORE_PREFIX}_degradation_row.csv"
deg_df.to_csv(deg_path, index=False)
print(f"Saved degradation row to {deg_path}")

display(deg_df)

## Section 6: Visualization

Generates 3 plots: CSR degradation curve, per-constraint-type bar chart, and constraint count distribution.
All plots are saved as PNGs to the results directory and displayed inline.

In [ ]:
from viz_utils import plot_csr_degradation, plot_per_type_bar, plot_constraint_distribution

# Plot 1: CSR Degradation Curve
plot_csr_degradation(metrics_by_level, MODEL_NAME, RESULTS_DIR)

# Plot 2: Per-Constraint-Type Bar Chart
plot_per_type_bar(per_type_rates, MODEL_NAME, RESULTS_DIR)

# Plot 3: Constraint Count Distribution
dist_data = [{"num_constraints": r["num_constraints"], "difficulty_level": r["difficulty_level"]} for r in results]
plot_constraint_distribution(dist_data, MODEL_NAME, RESULTS_DIR)

print(f"All plots saved to {RESULTS_DIR}")

## Section 7: Team Merge Utility

Scans the `scores/` directory for all `*_degradation_row.csv` files and merges them
into a single comparison table. Works with as few as one model evaluated.

In [ ]:
import glob

deg_files = glob.glob(os.path.join(SCORES_DIR, "*_degradation_row.csv"))

print(f"Found {len(deg_files)} degradation row file(s):")
for f in deg_files:
    print(f"  {f}")

if deg_files:
    all_rows = pd.concat([pd.read_csv(f) for f in deg_files], ignore_index=True)
    all_rows = all_rows.sort_values(["Model", "Run"], ascending=[True, False])

    print("\n=== Team Comparison ===")
    display(all_rows)

    comparison_path = os.path.join(SCORES_DIR, "team_comparison.csv")
    all_rows.to_csv(comparison_path, index=False)
    print(f"\nSaved to {comparison_path}")
else:
    print("No degradation rows found yet. Run evaluation for at least one model first.")

## Section 8: LLM Judge (Optional)

Uses Gemma-2-9B-Instruct as an LLM judge to evaluate qualitative constraints
that the rule-based checker could not handle (constraints where Section 4 returned `None`).
This covers tone, style, persona, and any other constraint types not in `ConstraintChecker`.

**VRAM warning:** The judge model requires ~6-9GB VRAM in 4-bit mode. If you ran
inference with a large model, the inference model should be unloaded first. This
section handles unloading automatically when the checkbox is checked.

Check the box below and run the remaining cells to enable.

In [ ]:
RUN_JUDGE_W = widgets.Checkbox(value=False, description="Run LLM judge")
JUDGE_BATCH_SIZE_W = widgets.IntSlider(value=1, min=1, max=4, description="Judge batch size:", style={'description_width': 'initial'})

display(widgets.VBox([RUN_JUDGE_W, JUDGE_BATCH_SIZE_W]))

In [ ]:
RUN_JUDGE = RUN_JUDGE_W.value
JUDGE_BATCH_SIZE = JUDGE_BATCH_SIZE_W.value

if not RUN_JUDGE:
    print("LLM judge skipped. Check the box above and re-run to enable.")
else:
    # Unload inference model if it exists
    if 'model' in dir() and model is not None:
        del model
        del tokenizer
        torch.cuda.empty_cache()
        print("Inference model unloaded.")

    from judge import JudgeModel

    judge = JudgeModel(use_4bit=USE_4BIT)

    # Load inference results
    responses_path = f"{RESULTS_DIR}/{MODEL_NAME_SAFE}_responses.jsonl"
    judge_checkpoint_path = f"{RESULTS_DIR}/{MODEL_NAME_SAFE}_judge_checkpoint.jsonl"
    judged_output_path = f"{RESULTS_DIR}/{MODEL_NAME_SAFE}_responses_judged.jsonl"

    inference_results = []
    with open(responses_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                inference_results.append(json.loads(line))
    print(f"Loaded {len(inference_results)} inference results.")

    # Re-run constraint checker to get rule-based results if not already present
    if inference_results and "results" not in inference_results[0]:
        from constraint_checker import ConstraintChecker
        checker = ConstraintChecker()
        for item in tqdm(inference_results, desc="Running rule-based checks"):
            eval_result = checker.check_all(item["response"], item["constraints"])
            item.update(eval_result)

    # Find records that have at least one None result
    needs_judging = [r for r in inference_results if any(cr["passed"] is None for cr in r.get("results", []))]
    print(f"Records with unjudged constraints: {len(needs_judging)}/{len(inference_results)}")

    # Load existing judge checkpoint
    judged_ids = set()
    judged_results = []
    if os.path.exists(judge_checkpoint_path):
        with open(judge_checkpoint_path, "r") as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    judged_ids.add(rec["id"])
                    judged_results.append(rec)
        print(f"Resumed from judge checkpoint: {len(judged_ids)} already judged.")

    remaining = [r for r in needs_judging if r["id"] not in judged_ids]
    print(f"Remaining to judge: {len(remaining)}")

    # Run judge
    checkpoint_file = open(judge_checkpoint_path, "a")
    try:
        for i, record in enumerate(tqdm(remaining, desc="LLM judging")):
            judged = judge.judge_all_skipped(record)
            judged_results.append(judged)
            judged_ids.add(judged["id"])
            checkpoint_file.write(json.dumps(judged) + "\n")

            if (i + 1) % 20 == 0:
                checkpoint_file.flush()
                print(f"  Judge checkpoint saved at {i + 1}/{len(remaining)}")
    finally:
        checkpoint_file.close()

    # Merge: use judged version for records that were judged, original for the rest
    judged_by_id = {r["id"]: r for r in judged_results}
    all_judged = []
    for r in inference_results:
        if r["id"] in judged_by_id:
            all_judged.append(judged_by_id[r["id"]])
        else:
            all_judged.append(r)

    # Save judged responses
    with open(judged_output_path, "w") as f:
        for r in all_judged:
            f.write(json.dumps(r) + "\n")
    print(f"Saved judged results to {judged_output_path}")

    # Recompute metrics on judged results
    by_level_j = defaultdict(list)
    for r in all_judged:
        if r.get("num_checked", 0) > 0:
            by_level_j[r["difficulty_level"]].append(r)

    metrics_judged = {}
    for level in ["L1", "L2", "L3", "L4"]:
        items = by_level_j.get(level, [])
        if items:
            csr = sum(r["per_constraint_csr"] for r in items) / len(items)
            hard = sum(1 for r in items if r["hard_csr"]) / len(items)
        else:
            csr, hard = 0.0, 0.0
        metrics_judged[level] = {"csr": round(csr, 4), "hard_csr": round(hard, 4), "count": len(items)}

    all_checked_j = [r for r in all_judged if r.get("num_checked", 0) > 0]
    ov_csr = sum(r["per_constraint_csr"] for r in all_checked_j) / len(all_checked_j) if all_checked_j else 0
    ov_hard = sum(1 for r in all_checked_j if r["hard_csr"]) / len(all_checked_j) if all_checked_j else 0
    metrics_judged["Overall"] = {"csr": round(ov_csr, 4), "hard_csr": round(ov_hard, 4), "count": len(all_checked_j)}

    # Save judged metrics
    judged_metrics_path = f"{SCORES_DIR}/{SCORE_PREFIX}_metrics_judged.json"
    with open(judged_metrics_path, "w") as f:
        json.dump({"model": MODEL_NAME, "run_timestamp": RUN_TIMESTAMP, "by_level": metrics_judged}, f, indent=2)
    print(f"Saved judged metrics to {judged_metrics_path}")

    # Save judged degradation row
    deg_row_j = {"Model": MODEL_NAME, "Run": RUN_TIMESTAMP, "Eval": "rule+judge"}
    for level in ["L1", "L2", "L3", "L4", "Overall"]:
        deg_row_j[f"{level}_CSR"] = metrics_judged[level]["csr"]
        deg_row_j[f"{level}_Hard"] = metrics_judged[level]["hard_csr"]
    deg_df_j = pd.DataFrame([deg_row_j])
    deg_j_path = f"{SCORES_DIR}/{SCORE_PREFIX}_degradation_row_judged.csv"
    deg_df_j.to_csv(deg_j_path, index=False)
    print(f"Saved judged degradation row to {deg_j_path}")

    # Side-by-side comparison
    print("\n=== Rule-Based vs Rule-Based + Judge ===")
    comparison_rows = []
    for level in ["L1", "L2", "L3", "L4", "Overall"]:
        rb = metrics_by_level.get(level, {"csr": 0, "hard_csr": 0})
        jd = metrics_judged.get(level, {"csr": 0, "hard_csr": 0})
        comparison_rows.append({
            "Level": level,
            "RuleBased_CSR": rb["csr"], "Judged_CSR": jd["csr"],
            "RuleBased_Hard": rb["hard_csr"], "Judged_Hard": jd["hard_csr"]
        })
    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)

    # Unload judge
    judge.unload()
    del judge
    print("Done.")